# A pydantic-ai agent that traces code through the graphlens MCP server

This notebook is a **standalone consumer** of graphlens — the kind of thing you
would build in *another* repo. graphlens itself only ever produces a graph; here
we let an LLM agent explore that graph at runtime through the
[Model Context Protocol](https://modelcontextprotocol.io) server shipped by
`graphlens-cli` (`graphlens mcp --graph graph.json`).

The MCP server exposes these tools, which the agent calls on its own to *trace*
relationships through the code graph:

| tool | what it returns |
|------|-----------------|
| `stats` | node/relation counts + resolver status |
| `find` | nodes matching a short or qualified name |
| `callers` | functions/methods that call a node |
| `callees` | functions/methods a node calls |
| `references` | nodes that reference a node |
| `neighbors` | nodes within N hops |
| `communicates_with` | cross-language `COMMUNICATES_WITH` edges |
| `boundaries` | cross-language boundary contracts |

## Requirements

```bash
pip install "graphlens-cli[all]" "pydantic-ai-slim[anthropic,mcp]"
export ANTHROPIC_API_KEY=sk-ant-...
jupyter lab
```

`graphlens-cli[all]` pulls in the language adapters and the `mcp` extra, so the
`graphlens` command is on your `PATH`. If you are running from a checkout of this
repo instead, use:

```bash
uv run --extra all --group notebook \
  --with 'pydantic-ai-slim[anthropic,mcp]' jupyter lab
```

## 1. Produce a graph for the agent to explore

The MCP server reads a graph JSON file (the same artifact `graphlens analyze
--output graph.json` writes). Here we analyze the graphlens repo itself as the
demo target, but point `TARGET_PROJECT` at any project you like.

In [ ]:
from pathlib import Path

from graphlens import adapter_registry

TARGET_PROJECT = Path("..").resolve()  # analyze the graphlens repo as a demo
GRAPH_PATH = Path("graph.json").resolve()

adapter = adapter_registry.load("python")()
graph = adapter.analyze(TARGET_PROJECT)
GRAPH_PATH.write_text(graph.to_json(indent=2), encoding="utf-8")

print(
    f"Wrote {GRAPH_PATH}\n"
    f"  {len(graph.nodes)} nodes / {len(graph.relations)} relations"
)

## 2. Wire the MCP server into a pydantic-ai agent

`MCPServerStdio` launches `graphlens mcp --graph graph.json` as a subprocess and
speaks MCP over stdio. We register it as a *toolset* on the agent, so every MCP
tool becomes a tool the model can call.

The default model is Claude Sonnet 4.6 (`claude-sonnet-4-6`) — a strong,
cost-effective choice for tool-calling loops. Swap in `claude-opus-4-8` for the
most capable option.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPServerStdio

# Needs ANTHROPIC_API_KEY in the environment.
MODEL = "anthropic:claude-sonnet-4-6"  # or "anthropic:claude-opus-4-8"

graphlens_mcp = MCPServerStdio(
    "graphlens",
    args=["mcp", "--graph", str(GRAPH_PATH)],
)

agent = Agent(
    MODEL,
    toolsets=[graphlens_mcp],
    system_prompt=(
        "You are a code-navigation assistant. Answer questions about a codebase "
        "using the graphlens MCP tools, which expose a precomputed code graph.\n"
        "Tools: stats, find, callers, callees, references, neighbors, "
        "communicates_with, boundaries.\n"
        "Workflow: call `find` to resolve a human-readable name to graph node "
        "ids, then use callers/callees/references/neighbors to trace "
        "relationships. Always cite the qualified_name and file_path of the "
        "nodes you report."
    ),
)

## 3. Ask the agent to trace something

`async with agent:` starts the MCP subprocess (and tears it down afterwards).
Jupyter supports top-level `await`, so we can run the coroutine directly.

In [ ]:
question = (
    "Which functions call GraphLens.add_node, and what does "
    "LanguageAdapter.analyze call? Trace it through the graph."
)

async with agent:
    result = await agent.run(question)

print(result.output)

## 4. Inspect the trace — every MCP tool call the agent made

This is the interesting part: `result.all_messages()` is the full transcript, so
we can see exactly which graph queries the agent issued and what came back. That
is the agent's *trace* over the graphlens MCP.

In [ ]:
def print_trace(result, *, max_preview: int = 300) -> None:
    """Pretty-print the agent's transcript: prompts, tool I/O, replies."""
    for message in result.all_messages():
        for part in message.parts:
            kind = getattr(part, "part_kind", type(part).__name__)
            if kind == "user-prompt":
                print(f"\U0001f464 user: {part.content}")
            elif kind == "tool-call":
                print(f"\U0001f527 call  {part.tool_name}({part.args})")
            elif kind == "tool-return":
                preview = str(part.content)
                if len(preview) > max_preview:
                    preview = preview[:max_preview] + " \u2026"
                print(f"   \u21a9 {part.tool_name} -> {preview}")
            elif kind == "thinking" and part.content.strip():
                print(f"\U0001f4ad {part.content.strip()[:max_preview]}")
            elif kind == "text" and part.content.strip():
                print(f"\U0001f916 {part.content.strip()}")


print_trace(result)

## 5. A reusable helper for more questions

Wrap the run + trace in one coroutine so you can keep probing the graph.

In [ ]:
async def ask(question: str):
    """Run the agent on a question and print both the answer and its trace."""
    async with agent:
        result = await agent.run(question)
    print(result.output)
    print("\n--- trace ---")
    print_trace(result)
    return result


_ = await ask(
    "Use the stats tool, then list the cross-language boundaries and who "
    "consumes them."
)